In [0]:
%pip install mysql-connector-python pyarrow

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 116.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import boto3

aws_access_key = dbutils.secrets.get("ecommerce", "aws_access_key_id")
aws_secret_key = dbutils.secrets.get("ecommerce", "aws_secret_access_key")

s3_client = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name="us-east-1"
)

bucket_name = "ecommerce-data-lake-jdavis2026"

In [0]:
bucket_name = 'ecommerce-data-lake-jdavis2026'

# List objects in the bucket
response = s3_client.list_objects_v2(Bucket=bucket_name, MaxKeys=10)
print(f"Objects in {bucket_name}:")
for obj in response.get('Contents', []):
    print(f"  - {obj['Key']}")

Objects in ecommerce-data-lake-jdavis2026:


In [0]:
import mysql.connector
import pandas as pd
import os

mysql_host = "ecommerce-db.cknwac2eed2a.us-east-1.rds.amazonaws.com"
mysql_user = dbutils.secrets.get("ecommerce", "mysql_user")
mysql_password = dbutils.secrets.get("ecommerce", "mysql_password")

run_date = "2026-07-30"  # update to today's actual date each time you run this
tables = ["customers", "products", "orders", "order_items", "inventory"]

conn = mysql.connector.connect(
    host=mysql_host,
    user=mysql_user,
    password=mysql_password,
    database="ecommerce"
)

for table in tables:
    print(f"Extracting {table}...")
    df = pd.read_sql(f"SELECT * FROM {table}", conn)

    local_path = f"/tmp/{table}.parquet"
    # Write with microsecond precision timestamps (Spark compatible)
    df.to_parquet(local_path, index=False, coerce_timestamps='us')

    s3_key = f"raw/{table}/dt={run_date}/{table}.parquet"
    s3_client.upload_file(local_path, bucket_name, s3_key)

    print(f"  {len(df)} rows -> s3://{bucket_name}/{s3_key}")

conn.close()
print("Extraction complete.")

Extracting customers...


/home/spark-6f3e8af8-ffac-4f11-83ee-fa/.ipykernel/73/command-7451566724412973-2650251536:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)


  505 rows -> s3://ecommerce-data-lake-jdavis2026/raw/customers/dt=2026-07-30/customers.parquet
Extracting products...
  200 rows -> s3://ecommerce-data-lake-jdavis2026/raw/products/dt=2026-07-30/products.parquet
Extracting orders...
  2000 rows -> s3://ecommerce-data-lake-jdavis2026/raw/orders/dt=2026-07-30/orders.parquet
Extracting order_items...
  4951 rows -> s3://ecommerce-data-lake-jdavis2026/raw/order_items/dt=2026-07-30/order_items.parquet
Extracting inventory...
  200 rows -> s3://ecommerce-data-lake-jdavis2026/raw/inventory/dt=2026-07-30/inventory.parquet
Extraction complete.


In [0]:
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix="raw/")
for obj in response.get('Contents', []):
    print(obj['Key'])

raw/customers/dt=2026-07-30/customers.parquet
raw/inventory/dt=2026-07-30/inventory.parquet
raw/order_items/dt=2026-07-30/order_items.parquet
raw/orders/dt=2026-07-30/orders.parquet
raw/products/dt=2026-07-30/products.parquet


In [0]:
aws_access_key = dbutils.secrets.get("ecommerce", "aws_access_key_id")
aws_secret_key = dbutils.secrets.get("ecommerce", "aws_secret_access_key")

bronze_db = "ecommerce_bronze"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")

tables = ["customers", "products", "orders", "order_items", "inventory"]
run_date = "2026-07-30"  # match what you used in the extract step

for table in tables:
    s3_path = f"s3a://{bucket_name}/raw/{table}/dt={run_date}/"
    df = spark.read \
        .option("fs.s3a.access.key", aws_access_key) \
        .option("fs.s3a.secret.key", aws_secret_key) \
        .parquet(s3_path)

    df = df.withColumn("_bronze_loaded_at", spark_functions.current_timestamp()) if False else df

    target_table = f"{bronze_db}.{table}"
    df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    print(f"{table}: {df.count()} rows -> {target_table}")

customers: 505 rows -> ecommerce_bronze.customers
products: 200 rows -> ecommerce_bronze.products
orders: 2000 rows -> ecommerce_bronze.orders
order_items: 4951 rows -> ecommerce_bronze.order_items
inventory: 200 rows -> ecommerce_bronze.inventory


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_db = "ecommerce_silver"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")

# ---- Clean customers ----
customers_bronze = spark.table("ecommerce_bronze.customers")

valid_tiers = ["BRONZE", "SILVER", "GOLD", "PLATINUM"]

customers_silver = (
    customers_bronze
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn(
        "loyalty_tier",
        F.when(F.upper(F.trim(F.col("loyalty_tier"))).isin(valid_tiers),
               F.upper(F.trim(F.col("loyalty_tier"))))
         .otherwise(F.lit("BRONZE"))
    )
    .withColumn(
        "is_email_valid",
        F.col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
    )
)

window = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
customers_silver = (
    customers_silver
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

customers_silver.write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.customers")
print(f"customers: {customers_silver.count()} rows -> {silver_db}.customers")

# ---- Clean orders ----
orders_bronze = spark.table("ecommerce_bronze.orders")

orders_cleaned = (
    orders_bronze
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("_dq_valid", (F.col("customer_id").isNotNull()) & (F.col("shipping_cost") >= 0))
)

window = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
orders_cleaned = (
    orders_cleaned
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

orders_valid = orders_cleaned.filter(F.col("_dq_valid") == True).drop("_dq_valid")
orders_quarantined = orders_cleaned.filter(F.col("_dq_valid") == False).drop("_dq_valid")

orders_valid.write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.orders")
orders_quarantined.write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.orders_quarantine")
print(f"orders: {orders_valid.count()} valid, {orders_quarantined.count()} quarantined")

# ---- Order items: compute line totals ----
order_items_bronze = spark.table("ecommerce_bronze.order_items")
order_items_silver = order_items_bronze.withColumn(
    "line_total",
    F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100.0), 2)
)
order_items_silver.write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.order_items")
print(f"order_items: {order_items_silver.count()} rows -> {silver_db}.order_items")

# ---- Products and inventory: pass through as-is ----
spark.table("ecommerce_bronze.products").write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.products")
spark.table("ecommerce_bronze.inventory").write.format("delta").mode("overwrite").saveAsTable(f"{silver_db}.inventory")
print("products and inventory copied to silver")

customers: 505 rows -> ecommerce_silver.customers
orders: 2000 valid, 0 quarantined
order_items: 4951 rows -> ecommerce_silver.order_items
products and inventory copied to silver


In [0]:
from datetime import date

business_key = "customer_id"
tracked_cols = ["loyalty_tier", "address", "city", "state", "postal_code", "email"]
gold_db = "ecommerce_gold"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

source_df = spark.table("ecommerce_silver.customers").select(
    business_key, *tracked_cols, "first_name", "last_name", "country"
)

def add_row_hash(df, cols):
    concat_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in cols])
    return df.withColumn("row_hash", F.sha2(concat_expr, 256))

as_of_date = date(2026, 7, 30)

hashed = add_row_hash(source_df, tracked_cols)
from pyspark.sql.window import Window as W
dim_customer = (
    hashed
    .withColumn("customer_sk", F.row_number().over(W.orderBy(business_key)))
    .withColumn("effective_start_date", F.lit(as_of_date))
    .withColumn("effective_end_date", F.lit(date(9999, 12, 31)))
    .withColumn("is_current", F.lit(True))
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{gold_db}.dim_customer")
print(f"dim_customer: {dim_customer.count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customer: 505 rows


In [0]:
orders_silver = spark.table("ecommerce_silver.orders")
order_items_silver = spark.table("ecommerce_silver.order_items")
dim_customer_current = spark.table("ecommerce_gold.dim_customer").filter(F.col("is_current") == True)

fact_sales = (
    order_items_silver.join(orders_silver, "order_id")
    .join(
        dim_customer_current.select("customer_id", "customer_sk", "loyalty_tier"),
        "customer_id"
    )
    .select(
        "order_id", "order_item_id", "customer_sk", "product_id",
        "order_date", "order_status", "quantity", "unit_price",
        "discount_pct", "line_total", "loyalty_tier"
    )
)

fact_sales.write.format("delta").mode("overwrite").saveAsTable("ecommerce_gold.fact_sales")
print(f"fact_sales: {fact_sales.count()} rows")

fact_sales: 4951 rows


In [0]:
display(
    spark.sql("""
        SELECT loyalty_tier, ROUND(SUM(line_total), 2) as revenue
        FROM ecommerce_gold.fact_sales
        GROUP BY loyalty_tier
        ORDER BY revenue DESC
    """)
)

loyalty_tier,revenue
BRONZE,1492849.51
PLATINUM,1482653.71
GOLD,1477940.76
SILVER,1205960.22
